# 数据格式

这个 notebook 说明本仓库使用的三类 JSONL 数据：SFT 数据、DPO 偏好数据、PPO/GRPO 奖励数据。

这里的样例不是占位符。`data/` 目录里的每一行都是真实 JSON 对象，可以被 `datasets.load_dataset("json", data_files=...)` 读取。

## 1. SFT 数据

SFT 数据用于让模型学习：看到 system 和 user 后，应该生成什么 assistant 回答。

核心字段是 `messages`。它是一个列表，每个元素包含 `role` 和 `content`。

- `system`：定义任务边界或回答风格。
- `user`：用户输入。
- `assistant`：训练目标，也就是模型应该学习生成的内容。

SFT、LoRA、QLoRA 都可以使用这种格式。区别不在数据，而在模型加载方式和训练参数。

In [ ]:
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files="../../data/sample_sft.jsonl", split="train")
sft_dataset[0]

## 2. DPO 数据

DPO 数据用于表达偏好。它不只告诉模型什么是正确回答，还告诉模型同一个问题下哪个回答更好、哪个回答更差。

核心字段：

- `prompt`：问题。
- `chosen`：更好的回答。
- `rejected`：更差的回答。

DPO 的价值在于不用先训练奖励模型，直接用成对偏好数据优化模型。

In [ ]:
dpo_dataset = load_dataset("json", data_files="../../data/sample_preference_dpo.jsonl", split="train")
dpo_dataset[0]

## 3. PPO/GRPO 奖励数据

PPO 和 GRPO 需要奖励信号。为了让新人能理解，本仓库优先使用可验证奖励：精确匹配、JSON 校验、正则匹配。

核心字段：

- `prompt`：模型要回答的问题。
- `answer`：标准答案或规则。
- `reward_type`：奖励函数类型。

In [ ]:
import json
import re

def score_response(response: str, answer: str, reward_type: str) -> float:
    text = response.strip()
    if reward_type == "exact_match":
        return 1.0 if text == answer else 0.0
    if reward_type == "regex_match":
        pattern = answer.removeprefix("regex:")
        return 1.0 if re.fullmatch(pattern, text) else 0.0
    if reward_type == "json_schema":
        try:
            expected = json.loads(answer)
            actual = json.loads(text)
        except json.JSONDecodeError:
            return 0.0
        return 1.0 if actual == expected else 0.0
    raise ValueError(f"unknown reward_type: {reward_type}")

rl_dataset = load_dataset("json", data_files="../../data/sample_prompts_rl.jsonl", split="train")
example = rl_dataset[0]
score_response("42", example["answer"], example["reward_type"])